In [ ]:
dataset = 'monk1'
data_root = './data'

In [ ]:
# === 2. IMPORTS AND UTILITIES ===
import torch
import numpy as np
import sys
import os
import requests
from torch.utils.data import DataLoader, TensorDataset
import torchvision
import torchvision.transforms as transforms

# Import SVM models and metrics
try:
    from models import SVCModel, SVRModel
    from sklearn.metrics import *
    from sklearn.multioutput import MultiOutputRegressor
    # Assuming data_loader.py functions are available/imported
    from utils.data_loader import get_monk1_data, get_ml_cup_data
    
except ImportError as e:
    print("FATAL ERROR: A required module was not found.")
    print(f"Error: {e}")
    # In Jupyter, we raise the error but don't exit the kernel
    
    
# --- Utility function to extract data from DataLoader to NumPy arrays ---
def extract_data_to_numpy(data_loader):
    """
    Converts data from a PyTorch DataLoader into a flattened NumPy array pair (X, y).
    Targets (y) are returned in their original dimensionality (e.g., [N, M] for multi-output).
    """
    X_list = []
    y_list = []
    for X, y in data_loader:
        # Flatten the input (e.g., 28x28 image -> 784 features)
        X_list.append(X.view(X.size(0), -1).numpy()) 
        # Convert labels to NumPy
        y_list.append(y.numpy())
    
    X_data = np.concatenate(X_list)
    y_data = np.concatenate(y_list)
    
    # We return y_data as is (2D array, e.g., [N, 1] or [N, M]).
    return X_data, y_data

In [ ]:
from utils.data_loader import get_monk1_data, get_ml_cup_data

# === 3. DATA LOADING AND PREPARATION ===
dataset_name = dataset
BATCH_SIZE = 1024

print(f"Loading dataset: {dataset_name.upper()}...")

# --- 3.1 Load Data Logic (Include all if/elif blocks for monk1, mlc25, mnist, fmnist, kmnist) ---    
print(f"Loading dataset: {dataset_name}...")
    
# Determine task type and load data (DataLoader objects are returned)
if dataset_name == 'monk1':
    train_loader, test_loader, INPUT_SIZE, OUTPUT_SIZE = get_monk1_data(BATCH_SIZE, data_root)
    is_regression_task = False
    metric_name = "Test Accuracy (%)"
elif dataset_name == 'mlc25':
    train_loader, validation_loader, test_loader, INPUT_SIZE, OUTPUT_SIZE = get_ml_cup_data(BATCH_SIZE, data_root)
    is_regression_task = True
    metric_name = "Test MEE"
else:
    # This block handles the error if the dataset is outside the specified choices (monk1, mlc25).
    print("Unsupported dataset for SVM.")
    sys.exit(1)


# --- 3.2 Data Preparation for Scikit-learn ---

# Convert DataLoaders (PyTorch) to NumPy arrays (Scikit-learn)
X_train, y_train = extract_data_to_numpy(train_loader)
X_test, y_test = extract_data_to_numpy(test_loader)

print(f"Data loaded: Training samples={X_train.shape[0]}, Test samples={X_test.shape[0]}")

In [ ]:
X_train.shape, y_train.shape

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, KFold
from sklearn.svm import SVC, SVR
from scipy.stats import loguniform
from sklearn.metrics import accuracy_score, mean_squared_error, classification_report


params = [
    # 1. Linear Kernel: C is the ONLY parameter. Gamma is implicitly 'scale' or ignored.
    {
        'kernel': ['linear'],
        'C': loguniform(1e-3, 1e2),
    },
    
    # 2. RBF/Poly Kernels with Discrete Gamma: Tests the two known heuristics.
    {
        'kernel': ['rbf', 'poly'],
        'C': loguniform(1e-3, 1e2),
        'gamma': ['scale', 'auto'], # Discrete strings only
    },
    
    # 3. RBF/Poly Kernels with Continuous Gamma: Randomly samples from the continuous range.
    {
        'kernel': ['rbf', 'poly'],
        'C': loguniform(1e-3, 1e2),
        'gamma': loguniform(1e-3, 1e1), # Continuous distribution object only
    },
]

kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42) if not is_regression_task else KFold(n_splits=5, shuffle=True, random_state=42)  

In [ ]:
# HP Randomized Search Configuration
base_estimator = SVC(max_iter=50000, cache_size=1000) if not is_regression_task \
            else SVR(max_iter=50000, cache_size=1000)

hp_search = RandomizedSearchCV(
    estimator = base_estimator,
    param_distributions = params,
    n_iter = 20,                     # Number of random configurations to try
    scoring = 'accuracy' if not is_regression_task else 'neg_mean_absolute_error',
    cv = kfold,
    n_jobs = -1,
    verbose = 1)                          # 5-Fold Cross-Validation


In [ ]:
if is_regression_task:
    final_model = MultiOutputRegressor(hp_search)  
else:
    final_model = hp_search

In [ ]:
final_model.fit(X_train, y_train)

In [ ]:
if is_regression_task:
    print(final_model.estimators_)
else:
    print(final_model.best_params_)
    

In [ ]:
# === 4. MODEL INITIALIZATION (FINAL RECONSTRUCTION) ===

# --- Funzione Helper (Invariata ma pulita per il tuo caso senza Pipeline) ---
def create_model_from_estimator(best_estimator):
    """
    Estrae i parametri dal best_estimator e crea una NUOVA istanza 
    del tuo wrapper personalizzato (SVCModel o SVRModel).
    """
    # 1. Recupera i parametri dal modello fittato
    params = best_estimator.get_params()
    
    # 2. Determina il tipo e istanzia il wrapper corretto
    if isinstance(best_estimator, SVC):
        return SVCModel(**params), 'svc'
    elif isinstance(best_estimator, SVR):
        return SVRModel(**params), 'svr'
    else:
        raise TypeError(f"Tipo non supportato per la ricostruzione: {type(best_estimator)}")

# --- 4.1 Logica di Ricostruzione Intelligente ---

final_models_list = []  # Qui salveremo i tuoi wrapper (sarà una lista di 1 o N elementi)
is_multi_output_result = False # Flag per ricordarci dopo come gestire la fit

# Caso A: Multi-Output (Regressione o Classificazione con più target)
# Controlliamo se final_model ha l'attributo 'estimators_', tipico dei wrapper MultiOutput
if hasattr(final_model, 'estimators_'):
    print(f"Detected Multi-Output System ({len(final_model.estimators_)} targets).")
    is_multi_output_result = True
    
    # Iteriamo su ogni RandomizedSearchCV contenuto nel MultiOutput
    for i, search_obj in enumerate(final_model.estimators_):
        # Estraiamo il vincitore per questo specifico target
        best_est = search_obj.best_estimator_
        
        # Creiamo il tuo wrapper
        wrapper, m_type = create_model_from_estimator(best_est)
        final_models_list.append(wrapper)
        
        print(f"Target {i}: Configurato {m_type.upper()} con C={wrapper.model.C:.4f}, gamma={wrapper.model.gamma}")

# Caso B: Singolo Output (Standard)
else:
    print("Detected Single-Output System.")
    # Qui final_model è direttamente la RandomizedSearchCV
    best_est = final_model.best_estimator_
    
    wrapper, m_type = create_model_from_estimator(best_est)
    final_models_list.append(wrapper)
    
    print(f"Configurato {m_type.upper()} con C={wrapper.model.C:.4f}, gamma={wrapper.model.gamma}")


# --- 4.2 Fit Finale e Predizione ---
# Ora final_models_list contiene i tuoi oggetti SVCModel/SVRModel NUOVI (resettati).
# Bisogna fare il fit finale sui dati di training.

print("\n--- Inizio Refitting dei modelli finali ---")

if is_multi_output_result:
    # Loop manuale per fittare ogni wrapper sulla sua colonna specifica
    for i, wrapper in enumerate(final_models_list):
        print(f"Fitting Target {i}...")
        # Nota: y_train[:, i] prende solo la colonna i-esima
        wrapper.model.fit(X_train, y_train[:, i]) 
        
else:
    # Caso singolo
    print("Fitting Single Model...")
    final_models_list[0].model.fit(X_train, y_train.ravel())

print("Refitting Completato.")

In [ ]:
def mean_euclidean_error(y_true, y_pred):
    errors = y_true - y_pred
    return np.linalg.norm(errors, axis=1).mean()

In [ ]:
from sklearn.metrics import make_scorer

mee_scorer = make_scorer(mean_euclidean_error, greater_is_better=False, needs_proba=False, needs_threshold=False)

In [ ]:
# === 5. TRAINING AND EVALUATION ===
import numpy as np

print("\n--- Starting SVM Training ---")

# Variabile per raccogliere le previsioni (serve per la valutazione dopo)
training_completed = False

# CASO A: Multi-Output (Abbiamo una lista di modelli, uno per target)
if is_multi_output_result:
    print(f"Modality: Multi-Output Independent Training ({len(final_models_list)} targets)")
    
    # Non esiste un 'model_container' unico. Fittiamo ogni wrapper sulla sua colonna.
    for i, wrapper in enumerate(final_models_list):
        print(f" -> Fitting Model for Target {i}...")
        
        # wrapper.model è l'oggetto SVR/SVC scikit-learn
        # y_train[:, i] prende solo la colonna specifica del target
        wrapper.model.fit(X_train, y_train[:, i])
        
# CASO B: Single-Output (Abbiamo una lista con 1 solo elemento)
else:
    print("Modality: Single-Output Training")
    
    # Prendiamo l'unico modello dalla lista
    model_container = final_models_list[0].model 
    
    # Prepariamo y con ravel()
    y_train_fit = y_train.ravel()
    
    # Fit standard
    model_container.fit(X_train, y_train_fit)

print("--- Training Completed ---")

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, mean_absolute_error

# === 6. PREDICTION & EVALUATION ===

print("\n--- Generating Predictions ---")

# 1. Generazione Predizioni (Gestisce il caso Lista vs Modello Singolo)
if is_multi_output_result:
    # Caso Multi-Output: Iteriamo sulla lista dei modelli
    column_preds = []
    for wrapper in final_models_list:
        pred = wrapper.model.predict(X_test)
        column_preds.append(pred)
    
    # Uniamo le colonne: (N_samples, N_targets)
    y_test_pred = np.column_stack(column_preds)
    
    # Assicuriamoci che y_test sia della forma corretta per il confronto
    y_test_eval = y_test 
    
else:
    # Caso Single-Output: Usiamo l'unico modello in lista
    y_test_pred = final_models_list[0].model.predict(X_test)
    
    # Appiattiamo y_test per coerenza (N,)
    y_test_eval = y_test.ravel()

print(f"Predictions generated. Shape: {y_test_pred.shape}")


# 2. Calcolo Metrica Finale
print("\n--- Calculating Metrics ---")

if is_regression_task:
    # Calculate MEE for Regression
    final_metric = mean_euclidean_error(y_test_eval, y_test_pred)
else:
    # Calculate Accuracy for Classification
    final_metric = accuracy_score(y_test_eval, y_test_pred) * 100.0
    
# 3. Risultato Finale
print(f"Final Test {metric_name}: {final_metric:.4f}")

In [ ]:
# --- ADDITIONAL VISUALIZATIONS AND METRICS ---

# 1) Classification
if not is_regression_task:
    try:
        y_scores = model_container.decision_function(X_test)
    except AttributeError:
        print("Model does not support decision_function; skipping ROC and AUC calculations.")
        y_scores = y_test_pred

    # Confusion Matrix
    cm = confusion_matrix(y_test_eval, y_test_pred)
    ConfusionMatrixDisplay(cm).plot()

    # ROC Curve
    fpr, tpr, _ = roc_curve(y_test_eval, y_scores)
    RocCurveDisplay(fpr = fpr, tpr = tpr).plot()

    # AUC Score
    auc_score = roc_auc_score(y_test_eval, y_scores) * 100.0
    print(f"AUC Score (%): {auc_score:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

def plot_search_results(search_object, target_name="Model", metric_name="Score"):
    """
    Visualizza i risultati della RandomizedSearchCV.
    Plotta C (log scale) vs Score.
    """
    # 1. Estraiamo i risultati in un DataFrame
    results_df = pd.DataFrame(search_object.cv_results_)
    
    # 2. Gestione Metrica (Se è negativa come neg_mae, la rendiamo positiva)
    # Scikit-learn usa punteggi negativi per le loss function
    score_col = 'mean_test_score'
    scores = results_df[score_col].values
    if scores.mean() < 0:
        scores = -scores # Convertiamo in errore positivo (es. MAE)
        ylabel = f"{metric_name} (Lower is better)"
    else:
        ylabel = f"{metric_name} (Higher is better)"
        
    # 3. Recuperiamo i parametri (C è il principale)
    # Cerchiamo la colonna che contiene il parametro C. 
    # Potrebbe chiamarsi 'param_C' o 'param_svm__C' (se usata pipeline)
    c_col = [col for col in results_df.columns if 'param_' in col and 'C' in col][-1]
    c_values = results_df[c_col].astype(float)
    
    # 4. Plot
    plt.figure(figsize=(10, 6))
    plt.scatter(c_values, scores, c='blue', edgecolors='k', s=70, alpha=0.7, label='Candidate Models')
    
    # Evidenziamo il migliore
    best_idx = search_object.best_index_
    plt.scatter(c_values[best_idx], scores[best_idx], c='red', s=150, marker='*', label='Best Estimator')
    
    plt.xscale('log') # C varia esponenzialmente, quindi scala logaritmica
    plt.xlabel('Parameter C (Regularization)')
    plt.ylabel(ylabel)
    plt.title(f'Validation Results: {target_name}')
    plt.grid(True, which="both", ls="-", alpha=0.2)
    plt.legend()
    plt.show()

# --- ESECUZIONE DEL PLOT ---

print("\n--- Plotting Validation Insights ---")

if is_multi_output_result:
    # Caso Multi-Output: Abbiamo una lista di search objects in final_model.estimators_
    # Ne plottiamo uno per target (o solo i primi due se sono tanti)
    n_targets_to_plot = min(len(final_model.estimators_), 3) # Limitiamo a 3 per non intasare
    
    for i in range(n_targets_to_plot):
        print(f"Plotting Target {i}...")
        plot_search_results(final_model.estimators_[i], 
                            target_name=f"Target Output {i}",
                            metric_name="Accuracy" if not is_regression_task else "MAE")
else:
    # Caso Single-Output
    # final_model è direttamente la RandomizedSearchCV (o dobbiamo recuperarla se abbiamo salvato solo il best)
    
    # ATTENZIONE: Se nel blocco precedente hai sovrascritto final_model con il best_estimator_,
    # devi usare l'oggetto 'hp_search' o 'hp_search_base' originale che contiene i risultati!
    
    # Assumo tu abbia ancora l'oggetto della search (es. hp_search_base o hp_search)
    if 'hp_search_base' in locals():
        search_obj_to_plot = hp_search_base
    elif 'hp_search' in locals():
        search_obj_to_plot = hp_search
    else:
        print("Errore: Oggetto RandomizedSearchCV originale non trovato in memoria.")
        search_obj_to_plot = None

    if search_obj_to_plot:
        plot_search_results(search_obj_to_plot, 
                            target_name="Single Output Model", 
                            metric_name="Accuracy" if not is_regression_task else "MAE")